# Jigsaw Puzzle Solver - System Showcase

This notebook demonstrates the complete jigsaw puzzle solving system with:

- **Success Cases**: Image 5 (2x2), Image 40 (4x4), Image 88 (8x8)
- **Failure Case**: Image 29 (4x4) with analysis
- **Intermediate Results**: Edge matching, candidate matches, similarity scores
- **Visualization**: Overlayed matched edges, connecting lines between candidate matches

## Setup Note

First, we'll preprocess the puzzle images to generate the required piece data (original, prep, upscaled, binary, edges, contours).


In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict, Tuple

# Add project root to path
sys.path.insert(0, os.path.abspath("."))

from utils.piece_loader import PieceLoader
from utils.similarity import SimilarityCalculator
from utils.image_utils import merge_pieces, load_image, split_image
from utils.preprocessing import preprocess
from utils.upscale import upscale_lanczos_sharp
from solvers.solver import solve

In [ ]:
def preprocess_puzzle(
    image_path: str, output_dir: Path, puzzle_id: int, grid_size: int
):
    """
    Preprocess a single puzzle image and generate all required outputs.
    """
    # Create output directories
    dirs = {
        "original": output_dir / "original",
        "prep": output_dir / "prep",
        "upscaled": output_dir / "upscaled",
        "binary": output_dir / "binary",
        "edges": output_dir / "edges",
        "contours": output_dir / "contours",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)

    # Load and split image
    image = load_image(image_path)
    piece_size = image.shape[0] // grid_size
    pieces = split_image(image, piece_size)

    # Process each piece
    for idx, piece in enumerate(pieces):
        row = idx // grid_size
        col = idx % grid_size
        base_name = f"puzzle_{puzzle_id:03d}_r{row}_c{col}.png"

        # 1. Original
        cv2.imwrite(
            str(dirs["original"] / base_name), cv2.cvtColor(piece, cv2.COLOR_RGB2BGR)
        )

        # 2. Preprocessed
        piece_prep = preprocess(piece, method="full")
        cv2.imwrite(
            str(dirs["prep"] / base_name), cv2.cvtColor(piece_prep, cv2.COLOR_RGB2BGR)
        )

        # 3. Upscaled
        piece_upscaled = upscale_lanczos_sharp(piece_prep, scale_factor=4)
        cv2.imwrite(
            str(dirs["upscaled"] / base_name),
            cv2.cvtColor(piece_upscaled, cv2.COLOR_RGB2BGR),
        )

        # 4. Binary
        gray = cv2.cvtColor(piece_upscaled, cv2.COLOR_RGB2GRAY)
        gray_prep = cv2.medianBlur(gray, 3)
        binary = cv2.adaptiveThreshold(
            gray_prep, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
        )
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
        cv2.imwrite(str(dirs["binary"] / base_name), binary)

        # 5. Edges
        v = np.median(gray_prep)
        low = int(0.55 * v)
        high = int(1 * v)
        edges = cv2.Canny(gray_prep, low, high)
        cv2.imwrite(str(dirs["edges"] / base_name), edges)

        # 6. Contours
        contours_data, _ = cv2.findContours(
            binary.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        contour_img = np.zeros_like(piece_upscaled)
        if contours_data:
            cv2.drawContours(contour_img, contours_data, -1, (255, 255, 255), 2)
        cv2.imwrite(
            str(dirs["contours"] / base_name),
            cv2.cvtColor(contour_img, cv2.COLOR_RGB2BGR),
        )

    print(
        f"✓ Preprocessed puzzle {puzzle_id} ({grid_size}x{grid_size}) -> {output_dir}"
    )


def ensure_puzzle_preprocessed(puzzle_id: int, grid_size: int, data_dir: str = "data"):
    """
    Check if puzzle is preprocessed, if not, preprocess it.
    Returns the output directory path.
    """
    size_name = f"{grid_size}x{grid_size}"
    output_dir = Path(f"output/tiles_{size_name}")

    # Check if already preprocessed
    expected_file = output_dir / "original" / f"puzzle_{puzzle_id:03d}_r0_c0.png"
    if expected_file.exists():
        return output_dir

    # Need to preprocess
    input_path = Path(data_dir) / f"puzzle_{size_name}" / f"{puzzle_id}.jpg"
    if not input_path.exists():
        raise FileNotFoundError(f"Input image not found: {input_path}")

    print(f"Preprocessing puzzle {puzzle_id} ({size_name})...")
    preprocess_puzzle(str(input_path), output_dir, puzzle_id, grid_size)
    return output_dir

## Preprocessing Functions

First, let's create functions to preprocess puzzle images if needed.


## Helper Functions for Visualization


In [ ]:
def visualize_edge_match(
    p1: np.ndarray, p2: np.ndarray, orientation: int, title: str = "Edge Match"
):
    """
    Visualize two pieces with their matching edges highlighted.
    orientation: 0=top, 1=bottom, 2=left, 3=right (where p2 is placed relative to p1)
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Show piece 1
    axes[0].imshow(p1)
    axes[0].set_title("Piece 1")
    axes[0].axis("off")

    # Show piece 2
    axes[1].imshow(p2)
    axes[1].set_title("Piece 2")
    axes[1].axis("off")

    # Show overlay of matching edges
    if orientation in [0, 1]:  # vertical
        if orientation == 0:  # p2 above p1
            edge1 = p1[:3, :]
            edge2 = p2[-3:, :]
        else:  # p2 below p1
            edge1 = p1[-3:, :]
            edge2 = p2[:3, :]
        combined = np.vstack([edge2, edge1])
    else:  # horizontal
        if orientation == 2:  # p2 left of p1
            edge1 = p1[:, :3]
            edge2 = p2[:, -3:]
        else:  # p2 right of p1
            edge1 = p1[:, -3:]
            edge2 = p2[:, :3]
        combined = np.hstack([edge2, edge1])

    axes[2].imshow(combined)
    axes[2].set_title("Edge Overlay")
    axes[2].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def draw_match_connections(
    pieces: List[np.ndarray],
    matches: List[Tuple[int, int, float]],
    grid_size: int,
    title: str = "Candidate Matches",
):
    """
    Draw pieces in a grid with lines connecting candidate matches.
    matches: List of (piece1_idx, piece2_idx, score)
    """
    piece_size = pieces[0].shape[0]
    canvas_size = grid_size * piece_size
    canvas = np.ones((canvas_size, canvas_size, 3), dtype=np.uint8) * 255

    # Draw all pieces
    for idx, piece in enumerate(pieces):
        row = idx // grid_size
        col = idx % grid_size
        y = row * piece_size
        x = col * piece_size
        canvas[y : y + piece_size, x : x + piece_size] = piece

    # Draw connection lines
    for p1_idx, p2_idx, score in matches:
        row1, col1 = p1_idx // grid_size, p1_idx % grid_size
        row2, col2 = p2_idx // grid_size, p2_idx % grid_size

        center1 = (
            col1 * piece_size + piece_size // 2,
            row1 * piece_size + piece_size // 2,
        )
        center2 = (
            col2 * piece_size + piece_size // 2,
            row2 * piece_size + piece_size // 2,
        )

        # Color based on score (green=good, red=bad)
        color_intensity = min(255, int(score / 100))
        color = (color_intensity, 255 - color_intensity, 0)

        cv2.line(canvas, center1, center2, color, 2)
        cv2.circle(canvas, center1, 5, color, -1)
        cv2.circle(canvas, center2, 5, color, -1)

    plt.figure(figsize=(12, 12))
    plt.imshow(canvas)
    plt.title(title)
    plt.axis("off")
    plt.show()


def compute_top_matches(
    pieces_dict: dict, sim_calc: SimilarityCalculator, top_k: int = 5
) -> Dict[int, List[Tuple[int, int, float]]]:
    """
    Compute top k matches for each piece in each orientation.
    Returns: {orientation: [(piece1, piece2, score), ...]}
    """
    n = len(pieces_dict["original"])
    matches_by_orientation = {0: [], 1: [], 2: [], 3: []}

    for orientation in range(4):
        scores = []
        for i in range(n):
            for j in range(n):
                if i != j:
                    score = sim_calc.compute(i, j, orientation, pieces_dict)
                    scores.append((i, j, score))

        # Sort by score (lower is better)
        scores.sort(key=lambda x: x[2])
        matches_by_orientation[orientation] = scores[:top_k]

    return matches_by_orientation


def display_puzzle_comparison(
    original_pieces: List[np.ndarray],
    shuffled_pieces: List[np.ndarray],
    solution: List[int],
    grid_size: int,
    title: str = "Puzzle Solution",
):
    """
    Display original, shuffled, and solved puzzle side by side.
    """
    original_img = merge_pieces(
        original_pieces, list(range(len(original_pieces))), grid_size
    )
    shuffled_img = merge_pieces(
        shuffled_pieces, list(range(len(shuffled_pieces))), grid_size
    )
    solved_img = merge_pieces(shuffled_pieces, solution, grid_size)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(original_img)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(shuffled_img)
    axes[1].set_title("Shuffled Pieces")
    axes[1].axis("off")

    axes[2].imshow(solved_img)
    axes[2].set_title("Solved Puzzle")
    axes[2].axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


def calculate_accuracy(solution: List[int], ground_truth: List[int]) -> float:
    """
    Calculate the percentage of correctly placed pieces.
    """
    correct = sum(1 for i, j in zip(solution, ground_truth) if i == j)
    return (correct / len(ground_truth)) * 100

## Initialize Similarity Calculator


In [ ]:
# Initialize similarity calculator with optimized weights
sim_calc = SimilarityCalculator(
    weight_color=5.0,
    weight_edge=0.8,
    weight_contour=1.0,
    color_depth=1,
    edge_depth=1,
    black_threshold=20,
)

print("✓ Similarity calculator initialized")

---

# Success Case 1: Image 5 (2x2 Puzzle)

Small puzzle with clear edge features.


In [ ]:
# Ensure puzzle is preprocessed and load it
puzzle_id_2x2 = 5
output_dir_2x2 = ensure_puzzle_preprocessed(puzzle_id_2x2, grid_size=2)

loader_2x2 = PieceLoader(str(output_dir_2x2), grid_size="2x2")
pieces_dict_2x2 = loader_2x2.load_all_types(puzzle_id_2x2)

print(f"Loaded puzzle {puzzle_id_2x2} (2x2)")
print(f"Available piece types: {list(pieces_dict_2x2.keys())}")
print(f"Number of pieces: {len(pieces_dict_2x2['original'])}")

In [ ]:
# Display shuffled pieces
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for idx, piece in enumerate(pieces_dict_2x2["original"]):
    row, col = idx // 2, idx % 2
    axes[row, col].imshow(piece)
    axes[row, col].set_title(f"Piece {idx}")
    axes[row, col].axis("off")
plt.suptitle("2x2 Puzzle - Shuffled Pieces (Image 5)")
plt.tight_layout()
plt.show()

### Intermediate Step: Edge Matching Analysis


In [ ]:
# Compute and display top candidate matches
print("Computing candidate matches for 2x2 puzzle...\n")
matches_2x2 = compute_top_matches(pieces_dict_2x2, sim_calc, top_k=3)

orientation_names = ["Top", "Bottom", "Left", "Right"]
for orientation, matches in matches_2x2.items():
    print(f"\nBest matches for {orientation_names[orientation]} edge:")
    for i, (p1, p2, score) in enumerate(matches, 1):
        print(f"  {i}. Piece {p1} <-> Piece {p2}: Score = {score:.2f}")

In [ ]:
# Visualize best match for right orientation (horizontal)
best_match = matches_2x2[3][0]  # Best right-edge match
p1_idx, p2_idx, score = best_match

visualize_edge_match(
    pieces_dict_2x2["original"][p1_idx],
    pieces_dict_2x2["original"][p2_idx],
    orientation=3,
    title=f"Best Horizontal Match: Piece {p1_idx} <-> Piece {p2_idx} (Score: {score:.2f})",
)

### Solve the Puzzle


In [ ]:
# Solve using genetic algorithm
print("Solving 2x2 puzzle with genetic algorithm...")
solution_2x2 = solve(
    pieces_dict_2x2,
    rows=2,
    cols=2,
    method="genetic",
    similarity=sim_calc,
    generations=50,
    population_size=50,
)

print(f"Solution: {solution_2x2}")
ground_truth = [0, 1, 2, 3]
accuracy = calculate_accuracy(solution_2x2, ground_truth)
print(f"Accuracy: {accuracy:.1f}%")

In [ ]:
# Display final result
display_puzzle_comparison(
    pieces_dict_2x2["original"],
    pieces_dict_2x2["original"],
    solution_2x2,
    grid_size=2,
    title=f"2x2 Puzzle Solution - Image 5 (Accuracy: {accuracy:.1f}%)",
)

---

# Success Case 2: Image 40 (4x4 Puzzle)

Medium complexity puzzle with good texture variation.


In [ ]:
# Ensure puzzle is preprocessed and load it
puzzle_id_4x4 = 40
output_dir_4x4 = ensure_puzzle_preprocessed(puzzle_id_4x4, grid_size=4)

loader_4x4 = PieceLoader(str(output_dir_4x4), grid_size="4x4")
pieces_dict_4x4 = loader_4x4.load_all_types(puzzle_id_4x4)

print(f"Loaded puzzle {puzzle_id_4x4} (4x4)")
print(f"Number of pieces: {len(pieces_dict_4x4['original'])}")

In [ ]:
# Display shuffled pieces
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for idx, piece in enumerate(pieces_dict_4x4["original"]):
    row, col = idx // 4, idx % 4
    axes[row, col].imshow(piece)
    axes[row, col].set_title(f"P{idx}", fontsize=8)
    axes[row, col].axis("off")
plt.suptitle("4x4 Puzzle - Shuffled Pieces (Image 40)")
plt.tight_layout()
plt.show()

### Intermediate Step: Candidate Matches Visualization


In [ ]:
# Compute top matches for horizontal connections
matches_4x4 = compute_top_matches(pieces_dict_4x4, sim_calc, top_k=8)

print("Top 8 horizontal (right-edge) matches for 4x4 puzzle:\n")
for i, (p1, p2, score) in enumerate(matches_4x4[3][:8], 1):
    print(f"  {i}. Piece {p1:2d} <-> Piece {p2:2d}: Score = {score:.2f}")

In [ ]:
# Draw candidate match connections
draw_match_connections(
    pieces_dict_4x4["original"],
    matches_4x4[3][:8],  # Top 8 horizontal matches
    grid_size=4,
    title="4x4 Puzzle - Top Horizontal Candidate Matches (Image 40)",
)

In [ ]:
# Visualize a few best edge matches
for i in range(2):
    p1_idx, p2_idx, score = matches_4x4[3][i]
    visualize_edge_match(
        pieces_dict_4x4["original"][p1_idx],
        pieces_dict_4x4["original"][p2_idx],
        orientation=3,
        title=f"Match #{i+1}: Piece {p1_idx} <-> Piece {p2_idx} (Score: {score:.2f})",
    )

### Solve the Puzzle


In [ ]:
# Solve using genetic algorithm
print("Solving 4x4 puzzle with genetic algorithm...")
solution_4x4 = solve(
    pieces_dict_4x4,
    rows=4,
    cols=4,
    method="genetic",
    similarity=sim_calc,
    generations=100,
    population_size=100,
)

print(f"Solution: {solution_4x4}")
ground_truth_4x4 = list(range(16))
accuracy_4x4 = calculate_accuracy(solution_4x4, ground_truth_4x4)
print(f"Accuracy: {accuracy_4x4:.1f}%")

In [ ]:
# Display final result
display_puzzle_comparison(
    pieces_dict_4x4["original"],
    pieces_dict_4x4["original"],
    solution_4x4,
    grid_size=4,
    title=f"4x4 Puzzle Solution - Image 40 (Accuracy: {accuracy_4x4:.1f}%)",
)

---

# Success Case 3: Image 88 (8x8 Puzzle)

Large puzzle demonstrating scalability of the system.


In [ ]:
# Ensure puzzle is preprocessed and load it
puzzle_id_8x8 = 88
output_dir_8x8 = ensure_puzzle_preprocessed(puzzle_id_8x8, grid_size=8)

loader_8x8 = PieceLoader(str(output_dir_8x8), grid_size="8x8")
pieces_dict_8x8 = loader_8x8.load_all_types(puzzle_id_8x8)

print(f"Loaded puzzle {puzzle_id_8x8} (8x8)")
print(f"Number of pieces: {len(pieces_dict_8x8['original'])}")

In [ ]:
# Display subset of shuffled pieces
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for idx in range(16):
    row, col = idx // 4, idx % 4
    axes[row, col].imshow(pieces_dict_8x8["original"][idx])
    axes[row, col].set_title(f"P{idx}", fontsize=8)
    axes[row, col].axis("off")
plt.suptitle("8x8 Puzzle - Sample of Shuffled Pieces (Image 88)")
plt.tight_layout()
plt.show()

### Intermediate Step: Match Analysis for Large Puzzle


In [ ]:
# Compute top matches
print("Computing candidate matches for 8x8 puzzle (this may take a moment)...\n")
matches_8x8 = compute_top_matches(pieces_dict_8x8, sim_calc, top_k=10)

print("Top 10 horizontal (right-edge) matches for 8x8 puzzle:\n")
for i, (p1, p2, score) in enumerate(matches_8x8[3][:10], 1):
    print(f"  {i:2d}. Piece {p1:2d} <-> Piece {p2:2d}: Score = {score:.2f}")

In [ ]:
# Visualize best edge matches from 8x8
for i in range(2):
    p1_idx, p2_idx, score = matches_8x8[3][i]
    visualize_edge_match(
        pieces_dict_8x8["original"][p1_idx],
        pieces_dict_8x8["original"][p2_idx],
        orientation=3,
        title=f"8x8 Best Match #{i+1}: Piece {p1_idx} <-> Piece {p2_idx} (Score: {score:.2f})",
    )

### Solve the Puzzle


In [ ]:
# Solve using genetic algorithm with more generations
print("Solving 8x8 puzzle with genetic algorithm (this will take longer)...")
solution_8x8 = solve(
    pieces_dict_8x8,
    rows=8,
    cols=8,
    method="genetic",
    similarity=sim_calc,
    generations=150,
    population_size=100,
)

print(f"Solution computed!")
ground_truth_8x8 = list(range(64))
accuracy_8x8 = calculate_accuracy(solution_8x8, ground_truth_8x8)
print(f"Accuracy: {accuracy_8x8:.1f}%")

In [ ]:
# Display final result
display_puzzle_comparison(
    pieces_dict_8x8["original"],
    pieces_dict_8x8["original"],
    solution_8x8,
    grid_size=8,
    title=f"8x8 Puzzle Solution - Image 88 (Accuracy: {accuracy_8x8:.1f}%)",
)

---

# Failure Case: Image 29 (4x4 Puzzle)

This image demonstrates a challenging case where the puzzle solver struggles.


In [ ]:
# Ensure puzzle is preprocessed and load it
puzzle_id_bad = 29
output_dir_bad = ensure_puzzle_preprocessed(puzzle_id_bad, grid_size=4)

pieces_dict_bad = PieceLoader(str(output_dir_bad), grid_size="4x4").load_all_types(
    puzzle_id_bad
)

print(f"Loaded puzzle {puzzle_id_bad} (4x4) - Challenging case")
print(f"Number of pieces: {len(pieces_dict_bad['original'])}")

In [ ]:
# Display shuffled pieces
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for idx, piece in enumerate(pieces_dict_bad["original"]):
    row, col = idx // 4, idx % 4
    axes[row, col].imshow(piece)
    axes[row, col].set_title(f"P{idx}", fontsize=8)
    axes[row, col].axis("off")
plt.suptitle("4x4 Puzzle - Challenging Case (Image 29)", color="red")
plt.tight_layout()
plt.show()

### Analysis: Why This Image Is Difficult


In [ ]:
# Compute match quality statistics
matches_bad = compute_top_matches(pieces_dict_bad, sim_calc, top_k=10)

# Compare with good image
print("=" * 60)
print("MATCH QUALITY COMPARISON")
print("=" * 60)

print("\n📊 Image 40 (GOOD) - Top 5 horizontal matches:")
scores_good = [score for _, _, score in matches_4x4[3][:5]]
for i, score in enumerate(scores_good, 1):
    print(f"  {i}. Score: {score:.2f}")
print(f"  Average: {np.mean(scores_good):.2f}")
print(f"  Std Dev: {np.std(scores_good):.2f}")

print("\n📊 Image 29 (BAD) - Top 5 horizontal matches:")
scores_bad = [score for _, _, score in matches_bad[3][:5]]
for i, score in enumerate(scores_bad, 1):
    print(f"  {i}. Score: {score:.2f}")
print(f"  Average: {np.mean(scores_bad):.2f}")
print(f"  Std Dev: {np.std(scores_bad):.2f}")

print("\n" + "=" * 60)

### Why Image 29 Fails - Detailed Analysis

**Reasons for poor performance:**

1. **Low Color Variation**: The image likely has uniform or homogeneous regions (e.g., sky, water, solid backgrounds) where many pieces look nearly identical. This makes it difficult to distinguish correct matches from incorrect ones.

2. **Weak Edge Features**: When pieces have similar colors across edges, the edge-matching algorithms cannot find strong discriminative features. Multiple pieces can appear to "match" equally well.

3. **High Ambiguity**: The similarity scores for different piece combinations are very close to each other (low standard deviation), meaning the algorithm cannot confidently distinguish between good and bad matches.

4. **Lack of Texture**: Without distinct textures, patterns, or gradients, the texture-based and gradient-based similarity measures provide little discriminative power.

5. **Black Borders**: If the image has significant black or dark borders after preprocessing, these get filtered out in color comparisons, leaving even less information to work with.

**What makes a good puzzle image:**

- Rich textures and patterns
- High color variation across regions
- Distinct gradients at edges
- Minimal uniform/solid color regions
- Clear structural elements (lines, shapes, objects)


In [ ]:
# Visualize ambiguous matches
print("\nVisualizing ambiguous matches from Image 29:\n")
for i in range(3):
    p1_idx, p2_idx, score = matches_bad[3][i]
    visualize_edge_match(
        pieces_dict_bad["original"][p1_idx],
        pieces_dict_bad["original"][p2_idx],
        orientation=3,
        title=f"Ambiguous Match #{i+1}: Piece {p1_idx} <-> Piece {p2_idx} (Score: {score:.2f})",
    )

### Attempt to Solve (Expected to Perform Poorly)


In [ ]:
# Solve the challenging puzzle
print("Attempting to solve challenging 4x4 puzzle...")
solution_bad = solve(
    pieces_dict_bad,
    rows=4,
    cols=4,
    method="genetic",
    similarity=sim_calc,
    generations=100,
    population_size=100,
)

print(f"Solution: {solution_bad}")
ground_truth_bad = list(range(16))
accuracy_bad = calculate_accuracy(solution_bad, ground_truth_bad)
print(f"\n❌ Accuracy: {accuracy_bad:.1f}% (Expected to be low)")

In [ ]:
# Display final result showing the failure
display_puzzle_comparison(
    pieces_dict_bad["original"],
    pieces_dict_bad["original"],
    solution_bad,
    grid_size=4,
    title=f"4x4 Puzzle - Failure Case - Image 29 (Accuracy: {accuracy_bad:.1f}%)",
)